# Task 1: Exploratory Data Analysis
**CM2604 Machine Learning: Coursework Part 01**

Explores the Telco Customer Churn dataset, visualises it and records the insights used in the report.

## Setup and data loading

In [ ]:
# Import all libraries used in this notebook
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay,
                             accuracy_score, classification_report,
                             f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TARGET = "Churn"

# Folders for saving figures and results (downloaded at the end)
FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
FIG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)


def save(fig, name):
    """Save a figure to figures/ and show it in the notebook."""
    fig.tight_layout()
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()

The dataset is loaded from the `data/` folder of the project's public GitHub repository.
**Change `GITHUB_USERNAME` below to your own GitHub username.**

In [ ]:
GITHUB_USERNAME = "yasiru20250101"         # your GitHub username
REPO_NAME = "telco-churn-classification"
DATA_URL = (f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/{REPO_NAME}"
            "/main/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

raw = pd.read_csv(DATA_URL)
print("Shape:", raw.shape)
raw.head()

## Data cleaning

In [ ]:
print("Data types:")
print(raw.dtypes)
print("\nMissing values (NaN):", raw.isna().sum().sum())

# TotalCharges is stored as text: check for blank strings
blank_tc = raw["TotalCharges"].astype(str).str.strip() == ""
print("Blank strings in TotalCharges:", blank_tc.sum())
print("Tenure of those customers:", raw.loc[blank_tc, "tenure"].unique())

In [ ]:
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]


def clean_data(df):
    """
    1. Drop customerID (unique identifier, no predictive value).
    2. Convert TotalCharges to numeric; blanks (tenure = 0) become NaN.
    3. Fill those NaN with 0 because new customers have not been charged yet.
    4. Convert SeniorCitizen 0/1 to No/Yes so it is treated as categorical.
    5. Encode the target: Yes -> 1, No -> 0.
    6. Remove duplicate rows.
    """
    df = df.copy()
    df = df.drop(columns=["customerID"])
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(),
                                       errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(0)
    df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
    df[TARGET] = df[TARGET].map({"No": 0, "Yes": 1})
    before = len(df)
    df = df.drop_duplicates()
    print(f"Duplicate rows removed: {before - len(df)}")
    return df


df = clean_data(raw)
print("Clean shape:", df.shape)
df[NUMERIC_FEATURES].describe().round(2)

### 1 Class distribution

In [ ]:
eda = df.copy()
eda["ChurnLabel"] = eda[TARGET].map({0: "No", 1: "Yes"})

print(eda["ChurnLabel"].value_counts())
print("Churn rate: %.2f%%" % (eda[TARGET].mean() * 100))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=eda, x="ChurnLabel", hue="ChurnLabel", legend=False, ax=ax)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}",
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
ax.set_title("Class distribution of Churn")
ax.set_xlabel("Churn")
save(fig, "01_class_distribution.png")

### 2 Numeric features vs churn

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(NUMERIC_FEATURES):
    sns.histplot(data=eda, x=col, hue="ChurnLabel", kde=True, element="step",
                 stat="density", common_norm=False, ax=axes[0, i])
    axes[0, i].set_title(f"Distribution of {col}")
    sns.boxplot(data=eda, x="ChurnLabel", y=col, hue="ChurnLabel",
                legend=False, ax=axes[1, i])
    axes[1, i].set_title(f"{col} by Churn")
save(fig, "02_numeric_vs_churn.png")

### 3 Churn rate by each categorical feature

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 18))
for ax, col in zip(axes.flatten(), CATEGORICAL_FEATURES):
    rate = eda.groupby(col)[TARGET].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.index, y=rate.values, ax=ax, color="steelblue")
    ax.set_title(f"Churn rate by {col}")
    ax.set_ylabel("Churn rate (%)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
save(fig, "03_churn_rate_by_category.png")

# Churn rates to quote in the report
for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport",
            "OnlineSecurity", "SeniorCitizen", "gender"]:
    print(f"\nChurn rate (%) by {col}:")
    print((eda.groupby(col)[TARGET].mean() * 100).round(1).to_string())

### 4 Contract type and tenure

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=eda, x="Contract", hue="ChurnLabel", ax=ax)
ax.set_title("Contract type vs Churn")
save(fig, "04_contract_vs_churn.png")

eda["TenureGroup"] = pd.cut(eda["tenure"], bins=[-1, 12, 24, 48, 72],
                            labels=["0-12", "13-24", "25-48", "49-72"])
rate = eda.groupby("TenureGroup", observed=True)[TARGET].mean() * 100
print("Churn rate (%) by tenure group:")
print(rate.round(1).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=rate.index, y=rate.values, ax=ax, color="indianred")
ax.set_title("Churn rate by tenure group (months)")
ax.set_ylabel("Churn rate (%)")
save(fig, "05_tenure_group_churn.png")

### 5 Correlation and tenure vs monthly charges

In [ ]:
corr = eda[NUMERIC_FEATURES + [TARGET]].corr()
print(corr.round(2))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation matrix")
save(fig, "06_correlation_heatmap.png")

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=eda, x="tenure", y="MonthlyCharges", hue="ChurnLabel",
                alpha=0.5, s=15, ax=ax)
ax.set_title("Tenure vs Monthly Charges coloured by Churn")
save(fig, "07_tenure_vs_monthly.png")

### EDA insights
*(Write your own observations here after running the cells above, e.g. which contract type, tenure group,
internet service and payment method have the highest churn.)*

## Download figures and results

In [ ]:
# Zip the figures/ and results/ folders and download them for the report
import shutil
shutil.make_archive("task1_figures", "zip", ".", "figures")
shutil.make_archive("task1_results", "zip", ".", "results")

try:
    from google.colab import files
    files.download("task1_figures.zip")
    files.download("task1_results.zip")
except ImportError:
    print("Not running in Colab: zip files saved in the current folder.")